# Notebook 06: Web Scraping of Job Postings (Workwise)

This notebook implements a small, legally uncontroversial web scraping module based on the method described by Dai et al. (2015).

Objective:
- Collect a few hundred current job postings from workwise.io (https://www.workwise.io/jobs)
- Expand the database of purely German external datasets
- Extract free-text fields (title, description, duties, requirements)
- Minimal cleaning
- Transfer data to the Unified Document Schema
- Save as Parquet

In [70]:
# Setup + Paths
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import uuid
import json

from pathlib import Path

# Project Root
PROJECT_ROOT = Path().resolve()
while not (PROJECT_ROOT / "data").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_RAW_EXTERNAL = PROJECT_ROOT / "data" / "raw_external"
DATA_INTERIM_EXTERNAL = PROJECT_ROOT / "data" / "interim_external"

# Create a folder
(DATA_RAW_EXTERNAL / "scraping").mkdir(parents=True, exist_ok=True)
DATA_INTERIM_EXTERNAL.mkdir(parents=True, exist_ok=True)

print("Pfade eingerichtet")

Pfade eingerichtet


## 1. Selecting the scraping source

workwise.io because:
- Job postings are publicly accessible
- No login barrier
- With moderate query frequency and no login barrier; scraping is limited to publicly accessible job postings and excludes personal data
- Clear HTML structure

Procedure:
1. https://www.workwise.io/jobs; November 25, 2025
2. Filter/Search (optional): Start with a broad search without filters for the first half of the URLs, then apply filters (for ArchiMate use cases) so that the profiles of the specific use case are covered as broadly as possible in the dataset. Search terms:
   - Mechanics/Mechatronics/Assembly/Maintenance
   - Operations Engineer/Mechanical Engineering/Technical Development
   - IT/Data/Azure/Data Lakehouse/Data Analysis
   - Sales (internal & external)/Customer Service/Sales Assistant
   - Accounting/Payroll/Administration
   - Manufacturing/Logistics/Procurement
3. Scroll through the first few pages
4. Copy the URLs of the individual job ads and paste them into the `job_urls` list. The file `raw_external/scraping/workwise_scraping-job-ads.txt` temporarily stores the URLs.

As described by Dai et al. (2015): Crawling individual profile pages using:
- requests.get() with User-Agent
- BeautifulSoup to extract title, description, and sections
- CSS selectors, robust
- Free-form text is consolidated

In [71]:
# List of job URLs manually added from the CSV file located at raw_external/job_ads/workwise_job_urls.csv
# Initial test with a few URLs
job_urls = [
    "https://www.workwise.io/job/53323-elektroniker-als-servicetechniker-m-w-d",
    "https://www.workwise.io/job/77464-bilanzbuchhalter-mit-fokus-steuererklaerung-m-w-d",
    "https://www.workwise.io/job/110163-bauingenieur-m-w-d",
    "https://www.workwise.io/job/112719-pflegefachkraft-m-w-d-fuer-den-nachtdienst-in-betzdorf",
    "https://www.workwise.io/job/112918-steuerberater-fuer-mittelstaendische-unternehmen-m-w-d",
    "https://www.workwise.io/job/113620-augenoptikermeister-fuer-kundenberatung-m-w-d",
    "https://www.workwise.io/job/113675-kalkulator-im-gleisbau-m-w-d",
    "https://www.workwise.io/job/113947-kfz-mechatroniker-fuer-nutzfahrzeuge-m-w-d",
    "https://www.workwise.io/job/114195-assistenz-der-geschaeftsleitung-in-der-immobilienbranche-m-w-d",
    "https://www.workwise.io/job/114248-anlagenmechaniker-fuer-heizungsanlagen-m-w-d",
    "https://www.workwise.io/job/114644-polier-fuer-verkehrsstationen-m-w-d",
    "https://www.workwise.io/job/114874-steuerfachangestellter-fuer-mandantenbetreuung-m-w-d",
    "https://www.workwise.io/job/114875-wirtschaftspruefer-und-steuerberater-fuer-hgb-abschluesse-m-w-d",
    "https://www.workwise.io/job/115033-steuerberater-fuer-mandantenbetreuung-m-w-d",
    "https://www.workwise.io/job/115165-store-manager-fuer-papeterie-einzelhandel-m-w-d",
    "https://www.workwise.io/job/115195-privatkundenberater-sparkasse-m-w-d",
    "https://www.workwise.io/job/115198-mitarbeiter-in-der-internen-revision-m-w-d",
    "https://www.workwise.io/job/115239-technischer-verkaufsberater-im-vertrieb-aussendienst-region-mitteldeutschland-m-w-d",
    "https://www.workwise.io/job/115389-inside-sales-manager-fuer-kundenbetreuung-m-w-d",
    "https://www.workwise.io/job/115447-bauleiter-im-bereich-verkehrssicherung-m-w-d",
    "https://www.workwise.io/job/115485-anwendungsberater-fuer-softwareloesungen-m-w-d",
    "https://www.workwise.io/job/115565-betreuungskraft-m-w-d",
    "https://www.workwise.io/job/115605-hausleitung-wohnparkleitung-ambulant-m-w-d-in-wangerland",
    "https://www.workwise.io/job/115751-mitarbeiter-qualitaetssicherung-auf-baustellen-m-w-d",
    "https://www.workwise.io/job/115756-steuerberater-w-m-d",
    "https://www.workwise.io/job/35454-junior-personalberater-fuer-active-sourcing-und-kommunikation-m-w-d",
    "https://www.workwise.io/job/89340-rechts-patentanwaltsfachangestellte-m-w-d",
    "https://www.workwise.io/job/99542-quereinsteiger-als-vertriebsmitarbeiter-fuer-neukundenakquise-m-w-d",
    "https://www.workwise.io/job/99547-projektmanager-energiewirtschaft-m-w-d",
    "https://www.workwise.io/job/99734-servicetechniker-fuer-kopier-und-drucksysteme-m-w-d",
    "https://www.workwise.io/job/105143-sap-entwickler-fuer-schnittstellenentwicklung-m-w-d",
    "https://www.workwise.io/job/105677-industriemechaniker-innen-und-aussenmonteur-m-w-d",
    "https://www.workwise.io/job/106015-personalreferent-fuer-aus-und-weiterbildung-m-w-d",
    "https://www.workwise.io/job/106554-mitarbeiter-im-vertriebsinnendienst-m-w-d",
    "https://www.workwise.io/job/108126-key-account-manager-w-m-d",
    "https://www.workwise.io/job/110398-ergotherapeut-m-w-d-in-der-rehabilitation",
    "https://www.workwise.io/job/112273-rechtsanwalt-rechtsanwaeltin-m-w-d-gesellschaftsrecht-ma",
    "https://www.workwise.io/job/113088-broker-fuer-key-accounts-m-w-d",
    "https://www.workwise.io/job/113464-third-level-support-im-managed-hosting-m-w-d",
    "https://www.workwise.io/job/113575-schichtkoordinator-verladung-m-w-d",
    "https://www.workwise.io/job/114247-steuerberater-mit-partneroption-m-w-d",
    "https://www.workwise.io/job/114294-pflegefachkraft-m-w-d-im-haus-drei-linden",
    "https://www.workwise.io/job/115063-oberflaechenbeschichter-in-der-industrielackierung-m-w-d",
    "https://www.workwise.io/job/115104-oberarzt-fuer-orthopaedie-und-endoprothetik-m-w-d",
    "https://www.workwise.io/job/115197-gewerbekundenberater-sparkasse-m-w-d",
    "https://www.workwise.io/job/115494-technischer-support-mitarbeiter-fuer-telekommunikation-m-w-d",
    "https://www.workwise.io/job/115523-consultant-fuer-unternehmensberatung-m-w-d",
    "https://www.workwise.io/job/115628-steuerberater-fuer-die-mandantenbetreuung-m-w-d",
    "https://www.workwise.io/job/115656-zweiradmechatronikermeister-fuer-fahrraeder-m-w-d-in-voll-oder-teilzeit",
    "https://www.workwise.io/job/115665-kreditsachbearbeiter-fuer-privat-und-gewerbekunden-m-w-d",
    "https://www.workwise.io/job/115679-referent-fuer-personalentwicklung-in-der-produktion-m-w-d",
    "https://www.workwise.io/job/115691-berater-manager-senior-manager-level-fuer-risikomanagement-m-w-d",
    "https://www.workwise.io/job/115697-pflegedienstleitung-im-ambulanten-dienst-m-w-d",
    "https://www.workwise.io/job/115781-syndikusrechtsanwalt-volljurist-m-w-d",
    "https://www.workwise.io/job/21249-consultant-fuer-unternehmensfinanzierung-projektfinanzierung-m-w-d",
    "https://www.workwise.io/job/77679-kundenberater-sparkasse-m-w-d",
    "https://www.workwise.io/job/101778-steuerberater-fuer-laufende-und-gestaltende-prozesse-m-w-d",
    "https://www.workwise.io/job/103364-pflegefachkraft-m-w-d-in-weilburg",
    "https://www.workwise.io/job/104427-pflegefachkraft-m-w-d-fuer-unseren-wohnpark-in-schortens",
    "https://www.workwise.io/job/105969-projektleiter-fuer-elektrotechnik-in-der-tga-m-w-d",
    "https://www.workwise.io/job/106646-junior-sales-consultant-in-innovationsberatung-m-w-d",
    "https://www.workwise.io/job/107731-systemmanager-fuer-sap-successfactors-projekte-m-w-d",
    "https://www.workwise.io/job/107889-senior-consultant-team-lead-energieeinkauf-m-w-d",
    "https://www.workwise.io/job/108840-rechtsanwalt-fuer-steuerrecht-beratung-m-w-d",
    "https://www.workwise.io/job/109502-ergotherapeut-m-w-d-in-betzdorf",
    "https://www.workwise.io/job/111046-facharzt-fuer-augenheilkunde-im-einzugsgebiet-m-w-d",
    "https://www.workwise.io/job/112335-betriebsingenieur-im-bereich-wasserstofftechnologie-m-w-d",
    "https://www.workwise.io/job/112948-kieferorthopaede-m-w-d-in-voll-oder-teilzeit",
    "https://www.workwise.io/job/113718-anlagenmechaniker-fuer-elektrolyseanwendungen-m-w-d",
    "https://www.workwise.io/job/113785-oberbauleiter-im-bereich-laermschutz-m-w-d",
    "https://www.workwise.io/job/114254-schutztechniker-fuer-mittel-und-niederspannung-m-w-d",
    "https://www.workwise.io/job/114507-technischer-leiter-fuer-pv-und-waermepumpen-m-w-d",
    "https://www.workwise.io/job/114701-maurer-m-w-d",
    "https://www.workwise.io/job/114709-ingenieur-fuer-sanitaertechnik-und-loeschanlagen-m-w-d",
    "https://www.workwise.io/job/114734-zahnmedizinische-fachangestellte-in-moderner-praxis-m-w-d",
    "https://www.workwise.io/job/114880-steuerberater-fuer-mandantenberatung-m-w-d",
    "https://www.workwise.io/job/114959-servicetechniker-fuer-gebaeudeautomation-m-w-d",
    "https://www.workwise.io/job/115008-vorarbeiter-fuer-dachdeckerarbeiten-m-w-d",
    "https://www.workwise.io/job/115010-leiter-bauwesen-facility-management-m-w-d",
    "https://www.workwise.io/job/115098-fachplaner-fuer-windenergieprojekte-m-w-d",
    "https://www.workwise.io/job/115220-volljurist-in-vollzeit-m-w-d",
    "https://www.workwise.io/job/115505-gutachter-im-bereich-marktfolge-aktiv-m-w-d",
    "https://www.workwise.io/job/115545-kauffrau-kaufmann-fuer-versicherung-und-finanzen-m-w-d-fuer-den-innendienst-voll-und-teilzeit",
    "https://www.workwise.io/job/115582-stellvertretende-pflegedienstleitung-ambulant-m-w-d-fuer-unseren-standort-in-betzdorf",
    "https://www.workwise.io/job/115657-verkaeufer-fuer-fahrraeder-und-zubehoer-m-w-d-in-voll-oder-teilzeit",
    "https://www.workwise.io/job/115729-steuerberater-m-w-d",
    "https://www.workwise.io/job/115693-steuerfachwirt-m-w-d",
    "https://www.workwise.io/job/115765-konstrukteur-fuer-inspektionskamerasysteme-m-w-d",
    "https://www.workwise.io/job/76206-steuerfachangestellter-in-teilzeit-m-w-d",
    "https://www.workwise.io/job/76235-steuerfachwirt-in-der-finanzbuchhaltung-in-teilzeit-m-w-d",
    "https://www.workwise.io/job/80424-maschinenfuehrer-fuer-die-zargenfertigung-mw-d",
    "https://www.workwise.io/job/106978-polier-im-tiefbau-und-gleisbau-m-w-d",
    "https://www.workwise.io/job/109555-innendienstmitarbeiter-im-technischen-vertrieb-m-w-d",
    "https://www.workwise.io/job/111116-elektrokonstrukteur-fuer-eplan-systeme-m-w-d",
    "https://www.workwise.io/job/111533-teamleiter-vertrieb-und-vertriebsstrategie-m-w-d",
    "https://www.workwise.io/job/114010-facharzt-fuer-gynaekologie-und-geburtshilfe-m-w-d",
    "https://www.workwise.io/job/114084-sachbearbeiter-im-kreditgeschaeft-m-w-d",
    "https://www.workwise.io/job/114173-kfz-nfz-meister-im-fahrzeug-und-karosseriebau-m-w-d",
    "https://www.workwise.io/job/114622-servicetechniker-fuer-sicherheitsalarmanlagen-m-w-d",
    "https://www.workwise.io/job/114640-zweiradmechaniker-kfz-mechatroniker-fuer-servicearbeiten-m-w-d",
    "https://www.workwise.io/job/115024-bauingenieur-fuer-projektmanagement-tragwerksplanung-m-w-d",
    "https://www.workwise.io/job/115034-steuerfachangestellte-fuer-finanzbuchhaltung-m-w-d",
    "https://www.workwise.io/job/115329-bilanzbuchhalter-m-w-d-in-der-insolvenzsteuerberatung",
    "https://www.workwise.io/job/115350-steuerberater-m-w-d-mit-aussicht-auf-beteiligung-bernahme",
    "https://www.workwise.io/job/115526-mechatroniker-m-w-d-umweltmesstechnik",
    "https://www.workwise.io/job/115546-steuerfachkraft-fuer-finanzbuchhaltung-und-beratung-m-w-d",
    "https://www.workwise.io/job/115601-hausleitung-wohnparkleitung-ambulant-m-w-d-in-varel",
    "https://www.workwise.io/job/115664-privatkundenberater-m-w-d",
    "https://www.workwise.io/job/115689-hardwareentwickler-fuer-embedded-systeme-m-w-d",
    "https://www.workwise.io/job/115698-pflegedienstleitung-in-der-altenpflege-m-w-d",
    "https://www.workwise.io/job/50503-steuerberater-tax-manager-m-w-d",
    "https://www.workwise.io/job/32967-baggerfuehrer-fuer-den-ganzjaehrigen-arbeitseinsatz-m-w-d",
    "https://www.workwise.io/job/100397-projektleiter-fuer-umspannwerke-planung-m-w-d",
    "https://www.workwise.io/job/103367-pflegefachkraft-m-w-d-in-wettenberg",
    "https://www.workwise.io/job/105017-immobilienkaufmann-fuer-bestandsimmobilien-m-w-d",
    "https://www.workwise.io/job/105694-bilanzbuchhalter-m-w-d",
    "https://www.workwise.io/job/108871-it-systemadministrator-fuer-vmware-und-storage-m-w-d",
    "https://www.workwise.io/job/109103-pflegefachkraft-als-praxisanleiter-m-w-d-fuer-unseren-standort-in-wetzlar",
    "https://www.workwise.io/job/109663-elektroniker-fuer-betriebstechnik-und-gebaeudetechnik-m-w-d",
    "https://www.workwise.io/job/111390-technischer-systemplaner-fuer-bauprojekte-m-w-d",
    "https://www.workwise.io/job/111395-empfangsmitarbeiter-fuer-patientenmanagement-m-w-d",
    "https://www.workwise.io/job/111836-elektriker-fuer-energie-gebaeudetechnik-m-w-d",
    "https://www.workwise.io/job/111951-projektleiter-in-veranstaltungstechnik-m-w-d",
    "https://www.workwise.io/job/112257-projektleitung-fuer-op-management-in-kliniken-m-w-d",
    "https://www.workwise.io/job/112271-notarfachangestellte-im-immobilienrecht-m-w-d",
    "https://www.workwise.io/job/112610-kfz-mechatroniker-fuer-die-montage-von-anhaengerkupplungen-m-w-d",
    "https://www.workwise.io/job/112797-elektroniker-fuer-automatisierungstechnik-m-w-d",
    "https://www.workwise.io/job/112826-kfz-mechatroniker-in-der-fahrzeugwartung-m-w-d",
    "https://www.workwise.io/job/113043-technischer-support-fuer-iot-geraete-m-w-d",
    "https://www.workwise.io/job/113483-dachdecker-m-w-d",
    "https://www.workwise.io/job/113535-senior-cal-al-developer-m-w-d",
    "https://www.workwise.io/job/113770-bankberater-im-privatkundengeschaeft-m-w-d",
    "https://www.workwise.io/job/113803-pflegefachkraft-m-w-d-fuer-unseren-ambulanten-dienst-in-limburg-gesucht",
    "https://www.workwise.io/job/113967-augenoptiker-geselle-augenoptik-m-w-d-in-zell-mosel",
    "https://www.workwise.io/job/114196-konstrukteur-formenbau-vorrichtungsbau-m-w-d",
    "https://www.workwise.io/job/114251-bau-facharbeiter-im-strassen-und-tiefbau-m-w-d",
    "https://www.workwise.io/job/114583-maler-und-lackierer-meister-im-handwerk-m-w-d",
    "https://www.workwise.io/job/114855-assistenz-der-bereichsleitung-elektroprojekte-m-w-d",
    "https://www.workwise.io/job/114864-kreativer-baecker-fuer-innovative-backwaren-m-w-d",
    "https://www.workwise.io/job/114966-hardware-entwickler-m-w-d",
    "https://www.workwise.io/job/115191-monteur-fuer-sonnenschutzanlagen-m-w-d",
    "https://www.workwise.io/job/115233-leitung-marktfolge-kredit-m-w-d",
    "https://www.workwise.io/job/115258-steuerfachangestellter-im-finanzwesen-m-w-d",
    "https://www.workwise.io/job/115352-zerspanungsmechaniker-fuer-fraestechnik-m-w-d",
    "https://www.workwise.io/job/115481-rohrvorrichter-in-berlin-marzahn-m-w-d",
    "https://www.workwise.io/job/115512-technischer-mitarbeiter-im-vertriebsinnendienst-m-w-d",
    "https://www.workwise.io/job/115577-leiter-vertriebs-und-servicebuero-kaeltetechnik-m-w-d",
    "https://www.workwise.io/job/45556-consultant-auditor-im-bereich-it-compliance-m-w-d",
    "https://www.workwise.io/job/67680-elektroniker-fuer-energie-und-gebaeudetechnik-im-shk-betrieb-m-w-d",
    "https://www.workwise.io/job/82283-privatkunden-berater-fuer-ganzheitliche-finanzberatung-m-w-d",
    "https://www.workwise.io/job/95599-sales-manager-im-b2b-vertrieb-m-w-d",
    "https://www.workwise.io/job/96085-technical-project-manager-m-w-d",
    "https://www.workwise.io/job/99722-berater-fuer-orthopaedieprodukte-m-w-d",
    "https://www.workwise.io/job/102337-heizungsbauer-anlagenmechaniker-shk-m-w-d",
    "https://www.workwise.io/job/103693-projektleiter-technische-gebaeudeausruestung-tga-m-w-d",
    "https://www.workwise.io/job/104432-land-und-baumaschinenmechatroniker-m-w-d",
    "https://www.workwise.io/job/104645-steuerfachangestellte-in-der-steuerberatung-m-w-d",
    "https://www.workwise.io/job/105326-senior-account-executive-existing-business-m-w-d",
    "https://www.workwise.io/job/105526-senior-sales-acquisition-manager-all-genders",
    "https://www.workwise.io/job/105654-wirtschaftspruefer-m-w-d-in-teilzeit-oder-vollzeit",
    "https://www.workwise.io/job/107587-service-techniker-im-aussendienst-m-w-d-pfalz",
    "https://www.workwise.io/job/108294-physiotherapeut-im-gesundheitswesen-m-w-d",
    "https://www.workwise.io/job/108550-consultant-im-kreditgeschaeft-fuer-baufinanzierung-m-w-d",
    "https://www.workwise.io/job/109503-logopaede-m-w-d-fuer-unseren-standort-in-betzdorf",
    "https://www.workwise.io/job/110696-meister-shk-m-w-d",
    "https://www.workwise.io/job/111917-vorarbeiter-fuer-flachdachabdichtung-m-w-d",
    "https://www.workwise.io/job/112264-technische-projektleitung-fuer-eventtechnik-m-w-d",
    "https://www.workwise.io/job/112296-pflegehelfer-pflegefachassistent-m-w-d",
    "https://www.workwise.io/job/112434-mechatroniker-oder-industriemechaniker-m-w-d",
    "https://www.workwise.io/job/112530-agrarkundenbetreuer-fuer-unser-bankhaus-w-m-d",
    "https://www.workwise.io/job/112634-steuerberater-fuer-buchfuehrung-und-abschluesse-m-w-d",
    "https://www.workwise.io/job/112650-lead-entwickler-fuer-softwareprojekte-m-w-d",
    "https://www.workwise.io/job/112677-steuerberater-fuer-buchfuehrung-m-w-d",
    "https://www.workwise.io/job/112701-elektromeister-m-w-d",
    "https://www.workwise.io/job/112890-elektroniker-fuer-maschinenbau-und-antriebstechnik-m-w-d",
    "https://www.workwise.io/job/113129-it-leiter-fuer-infrastrukturmanagement-m-w-d",
    "https://www.workwise.io/job/113164-bilanzbuchhalter-m-w-d",
    "https://www.workwise.io/job/113374-elektroniker-fuer-den-schaltschrankbau-m-w-d",
    "https://www.workwise.io/job/113398-verkaufsberater-fuer-kuechen-in-buchen-m-w-d",
    "https://www.workwise.io/job/113421-finanzbuchhalter-in-teilzeit-m-w-d",
    "https://www.workwise.io/job/113532-physiotherapeut-im-dynamischen-team-m-w-d",
    "https://www.workwise.io/job/113555-service-techniker-fuer-sprinkleranlagen-m-w-d",
    "https://www.workwise.io/job/113571-elektromonteur-fuer-mittel-und-niederspannung-m-w-d",
    "https://www.workwise.io/job/113574-elektroingenieur-im-gewerbebau-m-w-d",
    "https://www.workwise.io/job/113590-zimmerer-m-w-d",
    "https://www.workwise.io/job/114048-grafikdesigner-fuer-mediengestaltung-m-w-d",
    "https://www.workwise.io/job/114099-kfz-mechatroniker-fuer-nutzfahrzeuge-m-w-d",
    "https://www.workwise.io/job/114110-bauleiter-fuer-tga-projekte-m-w-d",
    "https://www.workwise.io/job/114116-fachkraft-fuer-explosionsschutz-und-anlagensicherheit-m-w-d",
    "https://www.workwise.io/job/114175-fahrzeuglackierer-fuer-lackierarbeiten-m-w-d",
    "https://www.workwise.io/job/114212-mitarbeiter-in-der-schallplattenproduktion-m-w-d",
    "https://www.workwise.io/job/114305-maschinenfuehrer-buchbinderei-in-der-druckverarbeitung-m-w-d",
    "https://www.workwise.io/job/114381-referent-fuer-finanzen-und-controlling-m-w-d",
    "https://www.workwise.io/job/114406-erp-administrator-fuer-sage-systeme-m-w-d",
    "https://www.workwise.io/job/114532-blechner-fuer-blechkonstruktionen-m-w-d",
    "https://www.workwise.io/job/114603-karosserie-und-fahrzeugbaumechaniker-fuer-nutzfahrzeuge-m-w-d",
    "https://www.workwise.io/job/114675-meister-dachdeckerhandwerk-obermonteur-dachdecker-m-w-d",
    "https://www.workwise.io/job/114787-bohrwerker-fuer-grossteilebearbeitung-m-w-d",
    "https://www.workwise.io/job/114844-dachdecker-fuer-dachdeckungen-und-daemmarbeiten-m-w-d",
    "https://www.workwise.io/job/114919-facharzt-aerztin-fuer-kinder-und-jugendmedizin-m-w-d",
    "https://www.workwise.io/job/114942-vermoegenskundenberater-sparkasse-regensburg-m-w-d",
    "https://www.workwise.io/job/115207-senior-kundenbetreuer-im-bereich-schifffahrt-m-w-d",
    "https://www.workwise.io/job/115219-mechaniker-fuer-baumaschinen-m-w-d",
    "https://www.workwise.io/job/115323-softwareentwickler-m-w-d-fuer-ki-systeme-und-infrastruktur",
    "https://www.workwise.io/job/115498-zimmerer-im-steildachbereich-m-w-d",
    "https://www.workwise.io/job/115548-lohnbuchhalter-m-w-d",
    "https://www.workwise.io/job/115625-kaufmaennischer-projektleiter-im-anlagenbau-m-w-d",
    "https://www.workwise.io/job/115790-middle-office-corporate-mitarbeiter-mit-kfw-foerderportal-kenntnissen-m-w-d",
    "https://www.workwise.io/job/35652-mitarbeiter-abrechnung-heiz-und-betriebskosten-m-w-d",
    "https://www.workwise.io/job/68142-sales-manager-im-b2b-vertrieb-fuer-it-hardware-m-w-d",
    "https://www.workwise.io/job/68801-bachelor-of-arts-steuern-und-pruefungswesen-m-w-d",
    "https://www.workwise.io/job/77310-elektromeister-m-w-d-mit-fuehrungsverantwortung",
    "https://www.workwise.io/job/89279-bauingenieur-fuer-verkehrsanlagen-tiefbau-m-w-d",
    "https://www.workwise.io/job/91229-bauingenieur-planung-verkehrsanlagen-m-w-d",
    "https://www.workwise.io/job/91929-lohnbuchhalter-m-w-d-in-der-insolvenzsteuerberatung",
    "https://www.workwise.io/job/99490-kaufmaennischer-mitarbeiter-im-technischen-innendienst-fuer-die-kundenberatung-m-w-d",
    "https://www.workwise.io/job/101421-mitarbeiter-fuer-arbeitsvorbereitung-und-datenmanagement-m-w-d",
    "https://www.workwise.io/job/102489-senior-performance-marketing-manager-fuer-meta-kampagnen-m-w-d",
    "https://www.workwise.io/job/103235-fachplaner-fuer-gebaeudeversorgungstechnik-hkls-m-w-d",
    "https://www.workwise.io/job/103368-pflegefachkraft-m-w-d-in-heuchelheim",
    "https://www.workwise.io/job/104565-leiter-fuer-ausschreibung-und-vergabe-m-w-d",
    "https://www.workwise.io/job/105655-bilanzbuchhalter-in-einer-steuerkanzlei-m-w-d-in-teilzeit-oder-vollzeit",
    "https://www.workwise.io/job/107989-schalungsbauer-fuer-betonfertigteile-m-w-d",
    "https://www.workwise.io/job/109491-projektingenieur-fuer-sanitaertechnikplanung-mit-schwerpunkt-druckluft-technische-gase-w-m-d",
    "https://www.workwise.io/job/109681-interims-buchhalter-m-w-d-in-festanstellung-oder-auf-selbststaendiger-basis",
    "https://www.workwise.io/job/110195-dreher-fuer-werkzeugbearbeitung-m-w-d",
    "https://www.workwise.io/job/110684-rechtsanwaltsfachangestellter-rechtsfachwirtin-legal-assistant-m-w-d",
    "https://www.workwise.io/job/110790-anwendungstechniker-in-der-papierindustrie-m-w-d",
    "https://www.workwise.io/job/111398-senior-consultant-fuer-sap-security-m-w-d",
    "https://www.workwise.io/job/111688-mitarbeiter-technisches-immobilienmanagement-m-w-d",
    "https://www.workwise.io/job/111828-elektromonteur-fuer-schaltgeraete-in-umspannwerken-m-w-d",
    "https://www.workwise.io/job/111967-projektant-projektverantwortlicher-elektro-oder-automatisierungstechnik-m-w-d",
    "https://www.workwise.io/job/112003-kundendienst-monteur-fuer-heizungstechnik-m-w-d",
    "https://www.workwise.io/job/112158-zimmerer-fuer-holz-und-blechkonstruktionen-m-w-d",
    "https://www.workwise.io/job/112919-pflegeassistenz-m-w-d-in-limburg",
    "https://www.workwise.io/job/113133-steuerberater-fuer-mandantenbetreuung-m-w-d",
    "https://www.workwise.io/job/113249-kfz-mechatroniker-fuer-nutzfahrzeuge-m-w-d",
    "https://www.workwise.io/job/113378-teamleiter-fuer-abrechnung-und-abfertigung-m-w-d",
    "https://www.workwise.io/job/113450-kfz-mechaniker-kfz-mechatroniker-m-w-d",
    "https://www.workwise.io/job/113460-maschineneinrichter-in-der-cnc-fertigung-m-w-d",
    "https://www.workwise.io/job/113639-zuschneider-im-bereich-flachglas-m-w-d",
    "https://www.workwise.io/job/113831-industrieelektriker-im-prototypenbau-m-w-d",
    "https://www.workwise.io/job/114115-wig-schweisser-im-prototypenbau-m-w-d",
    "https://www.workwise.io/job/114136-vorfuehrtechniker-fuer-baumaschinen-m-w-d",
    "https://www.workwise.io/job/114292-tiermedizinische-fachangestellte-m-w-d",
    "https://www.workwise.io/job/114371-rohrnetzmeister-fuer-gas-und-wasser-m-w-d",
    "https://www.workwise.io/job/114433-assistenz-des-cfo-m-w-d-remote",
    "https://www.workwise.io/job/114530-dachdecker-fuer-dachinspektionen-m-w-d",
    "https://www.workwise.io/job/114549-inside-sales-mitarbeiter-im-transportwesen-m-w-d",
    "https://www.workwise.io/job/114575-steuerfachwirt-in-einer-modernen-kanzlei-m-w-d",
    "https://www.workwise.io/job/114576-bilanzbuchhalter-in-einer-modernen-kanzlei-m-w-d",
    "https://www.workwise.io/job/114674-schlosser-fuer-metallbearbeitung-m-w-d",
    "https://www.workwise.io/job/114775-baecker-fuer-backwarenherstellung-m-w-d",
    "https://www.workwise.io/job/114865-konditor-fuer-backwaren-und-desserts-m-w-d",
    "https://www.workwise.io/job/114992-kfz-mechatroniker-m-w-d",
    "https://www.workwise.io/job/115120-personalsachbearbeiter-recruiter-m-w-d",
    "https://www.workwise.io/job/115060-leiter-bau-und-gebaeudemanagement-m-w-d",
    "https://www.workwise.io/job/115341-pflegeassistenz-m-w-d-fuer-unseren-wohnpark-in-zetel-gesucht",
    "https://www.workwise.io/job/115374-sharepoint-consultant-fuer-microsoft-365-loesungen-m-w-d",
    "https://www.workwise.io/job/115451-senior-sales-manager-fuer-b2b-automatisierung-m-w-d",
    "https://www.workwise.io/job/115596-finanzbuchhalter-m-w-d",
    "https://www.workwise.io/job/115602-technischer-planerin-fuer-stromversorgungsnetze-m-w-d",
    "https://www.workwise.io/job/115682-it-administrator-fuer-den-bereich-industrie-m-w-d",
    "https://www.workwise.io/job/115704-kaufmaennischer-mitarbeiter-m-w-d",
    "https://www.workwise.io/job/75145-junior-personalberater-active-sourcing-vertrieb-m-w-d",
    "https://www.workwise.io/job/81358-senior-bauzeichner-fuer-cad-zeichnungen-m-w-d",
    "https://www.workwise.io/job/81692-saas-sales-manager-m-w-d",
    "https://www.workwise.io/job/90230-bauueberwacher-fuer-kabelzug-und-baustellenkoordination-m-w-d",
    "https://www.workwise.io/job/94924-steuerberater-fuer-privatpersonen-m-w-d",
    "https://www.workwise.io/job/95074-bauleitung-hkls-m-w-d",
    "https://www.workwise.io/job/37537-anlagenfuehrer-fuer-die-maschinenbedienung-in-der-produktion-m-w-d",
    "https://www.workwise.io/job/115162-mechaniker-fuer-die-montage-im-anlagenbau-m-w-d",
    "https://www.workwise.io/job/100857-mechaniker-fuer-werkzeugmaschinen-im-aussendienst-m-w-d",
    "https://www.workwise.io/job/112337-projektleiter-fuer-mechatronik-projekte-m-w-d",
    "https://www.workwise.io/job/83851-elektroniker-fuer-unsere-rechenzentren-m-w-d",
    "https://www.workwise.io/job/108280-servicetechniker-fuer-metallbau-und-mechatronik-m-w-d",
    "https://www.workwise.io/job/110838-mechatroniker-nutzfahrzeuge-m-w-d",
    "https://www.workwise.io/job/114781-stellvertretender-abteilungsleiter-montage-m-w-d",
    "https://www.workwise.io/job/114791-mitarbeiter-in-der-montage-m-w-d",
    "https://www.workwise.io/job/97829-elektroniker-fuer-montage-und-reparatur-m-w-d",
    "https://www.workwise.io/job/114897-handwerker-im-montagebereich-m-w-d",
    "https://www.workwise.io/job/29354-elektriker-in-der-montage-m-w-d",
    "https://www.workwise.io/job/103131-elektroniker-fuer-fahrzeugausbau-und-montage-m-w-d",
    "https://www.workwise.io/job/109517-industrieelektriker-fuer-instandhaltung-m-w-d",
    "https://www.workwise.io/job/114267-mitarbeiter-instandhaltung-in-der-produktion-m-w-d",
    "https://www.workwise.io/job/106138-betriebselektriker-in-der-instandhaltung-m-w-d",
    "https://www.workwise.io/job/51099-elektroniker-im-bereich-instandhaltungstechnik-m-w-d",
    "https://www.workwise.io/job/115072-leiter-instandhaltung-im-industriellen-umfeld-m-w-d",
    "https://www.workwise.io/job/114034-technischer-einkaeufer-fuer-technik-und-instandhaltung-m-w-d",
    "https://www.workwise.io/job/105651-leitung-instandhaltung-fuer-gebaeude-und-anlagetechnikm-w-d",
    "https://www.workwise.io/job/112505-meister-fuer-elektrotechnik-in-der-instandhaltung-m-w-d",
    "https://www.workwise.io/job/75081-mechaniker-im-bereich-instandhaltungstechnik-m-w-d",
    "https://www.workwise.io/job/103650-anlagenmonteur-im-aussendienst-m-w-d",
    "https://www.workwise.io/job/110774-kundendienstmonteur-fuer-hlsk-anlagen-m-w-d",
    "https://www.workwise.io/job/104715-servicemonteur-fuer-tuer-und-toranlagen-m-w-d",
    "https://www.workwise.io/job/55409-anlagenmechaniker-shk-fuer-kundendienst-m-w-d",
    "https://www.workwise.io/job/93738-anlagenmechaniker-shk-m-w-d",
    "https://www.workwise.io/job/106206-anlagenmechaniker-shk-im-kundendienst-m-w-d",
    "https://www.workwise.io/job/19268-anlagenmechaniker-shk-technik-m-w-d",
    "https://www.workwise.io/job/112458-anlagenmechaniker-shk-in-bauleitender-funktion-m-w-d",
    "https://www.workwise.io/job/109172-anlagenmechaniker-fuer-shk-m-w-d",
    "https://www.workwise.io/job/112095-industriemechaniker-fuer-digitale-produktionsprozesse-m-w-d",
    "https://www.workwise.io/job/105959-industriemechaniker-fuer-produktionsanlagen-m-w-d",
    "https://www.workwise.io/job/110045-industriemechaniker-fuer-maschinenwartung-und-instandhaltung-m-w-d",
    "https://www.workwise.io/job/102265-mechatroniker-fuer-industrieanlagen-technik-m-w-d",
    "https://www.workwise.io/job/114220-industriemechaniker-fuer-anlagenmontage-m-w-d",
    "https://www.workwise.io/job/115509-industriemechaniker-fuer-montage-und-logistik-m-w-d",
    "https://www.workwise.io/job/109413-industriemechaniker-m-w-d",
    "https://www.workwise.io/job/112422-mechaniker-fuer-externe-montageprojekte-m-w-d",
    "https://www.workwise.io/job/59292-quereinsteiger-als-wartungstechniker-m-w-d",
    "https://www.workwise.io/job/91991-monteur-fuer-den-kundendienst-m-w-d",
    "https://www.workwise.io/job/104284-monteur-fuer-aluminium-bauelemente-m-w-d",
    "https://www.workwise.io/job/112884-monteur-fuer-sicherheitssysteme-m-w-d",
    "https://www.workwise.io/job/109550-elektroniker-fuer-betriebstechnik-m-w-d",
    "https://www.workwise.io/job/111982-elektroniker-fuer-die-betriebstechnik-m-w-d",
    "https://www.workwise.io/job/109634-elektroniker-fuer-betriebstechnik-in-der-produktion-m-w-d",
    "https://www.workwise.io/job/100565-elektroniker-fuer-betriebstechnik-m-w-d",
    "https://www.workwise.io/job/113966-elektroniker-fuer-betriebstechnik-m-w-d",
    "https://www.workwise.io/job/115497-elektroniker-fuer-betriebstechnik-m-w-d",
    "https://www.workwise.io/job/107962-elektroniker-fuer-betriebstechnik-und-wartung-m-w-d",
    "https://www.workwise.io/job/111922-industrieelektriker-fuer-betriebstechnik-m-w-d",
    "https://www.workwise.io/job/15617-elektroniker-im-anlagenbau-sps-fehleranalyse-m-w-d",
    "https://www.workwise.io/job/94525-techniker-fuer-kaeltetechnik-projekte-m-w-d",
    "https://www.workwise.io/job/94529-techniker-fuer-sanitaer-heizung-und-klimatechnik-m-w-d",
    "https://www.workwise.io/job/76090-technischer-planer-mit-fokus-elektrotechnik-m-w-d",
    "https://www.workwise.io/job/109435-maschinenfuehrer-fuer-extrusionstechnik-m-w-d",
    "https://www.workwise.io/job/115064-maschinenfuehrer-in-der-blechverarbeitung-m-w-d",
    "https://www.workwise.io/job/113361-maschinen-anlagenfuehrer-m-w-d",
    "https://www.workwise.io/job/111419-sps-programmierer-simatic-step-7-tia-portal-m-w-d",
    "https://www.workwise.io/job/110624-projektingenieur-im-anlagenbau-m-w-d",
    "https://www.workwise.io/job/115419-techniker-im-service-und-kalibrierlabor-m-w-d",
    "https://www.workwise.io/job/105918-techniker-ingenieur-m-w-d-schwerpunkt-projektabwicklung",
    "https://www.workwise.io/job/106225-cyber-defense-berater-m-w-d",
    "https://www.workwise.io/job/114836-senior-accountant-m-w-d",
    "https://www.workwise.io/job/111861-ingenieur-techniker-fuer-gebaeudetechnik-tga-m-w-d",
    "https://www.workwise.io/job/94524-meister-shk-und-bauleiter-fuer-gebaeudetechnik-m-w-d",
    "https://www.workwise.io/job/110240-techniker-sanitaerhandwerk-w-m-d",
    "https://www.workwise.io/job/105391-kaelteanlagenbauer-fuer-klimatechnik-m-w",
    "https://www.workwise.io/job/109483-klimatechniker-fuer-shk-m-w-d",
    "https://www.workwise.io/job/114659-technischer-produktdesigner-fuer-maschinenbau-m-w-d",
    "https://www.workwise.io/job/97649-aussendienstmonteur-fuer-maschinenbau-m-w-d",
    "https://www.workwise.io/job/112725-ausbildung-zum-zerspanungsmechaniker-im-maschinenbau-m-w-d",
    "https://www.workwise.io/job/105147-technische-leitung-fuer-gebaeudetechnik-m-w-d",
    "https://www.workwise.io/job/108465-technischer-spezialist-fuer-entwicklungsprojekte-m-w-d",
    "https://www.workwise.io/job/25716-softwareentwickler-kuenstliche-intelligenz-ki-m-w-d",
    "https://www.workwise.io/job/110499-senior-software-entwickler-fuer-kreislaufwirtschaft-m-w-d",
    "https://www.workwise.io/job/111744-projekt-und-teamleitung-im-bereich-shk-m-w-d",
    "https://www.workwise.io/job/104339-projektleiter-im-bereich-wasserbau-m-w-d",
    "https://www.workwise.io/job/115839-kaufmaennischer-mitarbeiter-im-controlling-und-it-m-w-d",
    "https://www.workwise.io/job/112496-wirtschaftsjurist-fuer-insolvenzverwaltung-m-w-d",
    "https://www.workwise.io/job/105393-senior-business-development-manager-m-w-d-in-der-energiebranche",
    "https://www.workwise.io/job/115338-technischer-vertriebsmitarbeiter-im-aussendienst-m-w-d-teilgebiete-schleswig-holstein",
    "https://www.workwise.io/job/114977-technischer-mitarbeiterin-abrechnung-und-aufmass-m-w-d",
    "https://www.workwise.io/job/15615-servicetechniker-im-bereich-pumpentechnik-m-w-d",
    "https://www.workwise.io/job/110514-qualitaetsvorausplaner-m-w-d",
    "https://www.workwise.io/job/107595-qualitaetsingenieur-m-w-d",
    "https://www.workwise.io/job/100751-qualitaetsmanager-schwerpunkt-futtermittelbereich-m-w-d",
    "https://www.workwise.io/job/111690-qualitaetsmanager-m-w-d",
    "https://www.workwise.io/job/115074-qualitaetsleiter-fuer-qualitaetsmanagement-m-w-d",
    "https://www.workwise.io/job/112983-qualitaetsmanager-im-maschinenbau-m-w-d",
    "https://www.workwise.io/job/95701-leiter-fuer-qualitaets-und-umweltmanagement-m-w-d",
    "https://www.workwise.io/job/114845-qualitaetsmanager-in-der-produktion-m-w-d",
    "https://www.workwise.io/job/100022-mitarbeiter-qualitaetssicherung-m-w-d",
    "https://www.workwise.io/job/114458-quality-manager-fuer-qualitaetsrichtlinien-m-w-d",
    "https://www.workwise.io/job/106016-sachbearbeiter-im-produktmanagement-m-w-d",
    "https://www.workwise.io/job/110074-entwicklungsingenieur-in-der-medizintechnik-m-w-d",
    "https://www.workwise.io/job/86949-entwicklungsingenieur-fuer-mechanikkonstruktion-m-w-d",
    "https://www.workwise.io/job/82312-entwicklungsingenieur-in-der-lasertechnik-m-w-d",
    "https://www.workwise.io/job/115216-entwicklungsingenieur-konstruktion-m-w-d",
    "https://www.workwise.io/job/86309-entwicklungsingenieur-fuer-einspritzsysteme-m-w-d",
    "https://www.workwise.io/job/114540-entwicklungsingenieur-fuer-elektronikdesign-m-w-d",
    "https://www.workwise.io/job/110662-manufacturing-engineer-fuer-prozessoptimierung-m-w-d",
    "https://www.workwise.io/job/109285-einrichter-fuer-produktionsanlagen-m-w-d",
    "https://www.workwise.io/job/113624-extrusionsingenieur-fuer-prozessoptimierung-m-w-d",
    "https://www.workwise.io/job/112907-projektplaner-ac-fuer-photovoltaikanlagen-im-bereich-privatkunden-m-w-d",
    "https://www.workwise.io/job/100028-projektplaner-fuer-gebaeudeinstallation-m-w-d",
    "https://www.workwise.io/job/45819-projektleiter-fuer-fachplanung-im-bereich-elektrotechnik-m-w-d",
    "https://www.workwise.io/job/112937-produktionsplaner-m-w-d",
    "https://www.workwise.io/job/101383-projektleiter-m-w-d",
    "https://www.workwise.io/job/104192-projektleiter-im-ingenieurbau-fuer-gfk-projekte-m-w-d",
    "https://www.workwise.io/job/111587-projektleiter-elektrotechnik-m-w-d",
    "https://www.workwise.io/job/111843-technical-communication-specialist-m-w-d",
    "https://www.workwise.io/job/92067-technischer-redakteur-fuer-software-dokumentation-m-w-d",
    "https://www.workwise.io/job/112896-technical-writer-specialist-systems-software-documentation-w-m-d",
    "https://www.workwise.io/job/103109-technischer-redakteur-im-bereich-softwaredokumentation-automatisierung-m-w-d",
    "https://www.workwise.io/job/113037-mitarbeiter-fuer-technische-dokumentation-und-produktkonformitaet-m-w-d",
    "https://www.workwise.io/job/111099-qualitaetsingenieur-fuer-ce-koordination-m-w-d",
    "https://www.workwise.io/job/100885-ingenieur-fuer-medienversorgung-und-qualitaetssicherung-m-w-d",
    "https://www.workwise.io/job/100812-verfahrenstechniker-im-anlagenbau-fuer-umwelttechnik-m-w-d",
    "https://www.workwise.io/job/107872-ingenieur-fuer-versorgungstechnik-m-w-d",
    "https://www.workwise.io/job/111954-vermessungsingenieur-m-w-d",
    "https://www.workwise.io/job/99630-senior-it-pruefer-it-berater-m-w-d",
    "https://www.workwise.io/job/52287-it-system-engineer-fuer-einen-modern-workplace-m-w-d",
    "https://www.workwise.io/job/115333-it-support-spezialist-fuer-logistik-it-m-w-d",
    "https://www.workwise.io/job/111364-it-system-kaufmann-fuer-it-management-m-w-d",
    "https://www.workwise.io/job/109247-it-consultant-m-w-d-it-security-netzwerke-infrastruktur",
    "https://www.workwise.io/job/108776-it-systemingenieur-fuer-it-projekte-m-w-d",
    "https://www.workwise.io/job/113756-it-administrator-fuer-it-infrastruktur-m-w-d",
    "https://www.workwise.io/job/101264-it-consultant-iga-m-w-d",
    "https://www.workwise.io/job/108199-senior-it-systemingenieur-m-w-d",
    "https://www.workwise.io/job/114772-it-manager-m-w-d",
    "https://www.workwise.io/job/114326-it-manager-fuer-regulatorik-m-w-d",
    "https://www.workwise.io/job/113089-fibutroniker-fachassistent-digitalisierung-und-it-it-spezialist-datevm-w-d",
    "https://www.workwise.io/job/112933-senior-it-recruiter-m-w-d",
    "https://www.workwise.io/job/112167-vermessungstechniker-m-w-d",
    "https://www.workwise.io/job/59739-senior-consultant-im-datenschutz-management-in-voll-und-teilzeit-m-w-d",
    "https://www.workwise.io/job/115741-cad-datenexperte-fuer-3d-automodelle-m-w-d",
    "https://www.workwise.io/job/99963-datenbank-architekt-fuer-codemeter-license-central-m-w-d",
    "https://www.workwise.io/job/115335-finanzbuchhalter-im-bereich-datev-m-w-d",
    "https://www.workwise.io/job/31172-finanzbuchhalter-fuer-datev-m-w-d",
    "https://www.workwise.io/job/111009-online-marketing-manager-mit-fokus-meta-ads-m-w-d",
    "https://www.workwise.io/job/107029-senior-data-architekt-fuer-dateninfrastruktur-m-w-d",
    "https://www.workwise.io/job/113096-werkstudentin-fuer-business-data-analyse-m-w-d",
    "https://www.workwise.io/job/115080-power-bi-spezialist-fuer-dashboard-entwicklung-m-w-d",
    "https://www.workwise.io/job/114744-werksassistenz-fuer-organisation-und-verwaltung-m-w-d",
    "https://www.workwise.io/job/85093-werkstudent-fuer-datenbankpflege-m-w-d",
    "https://www.workwise.io/job/112652-cloud-entwickler-fuer-aws-und-azure-projekte-m-w-d",
    "https://www.workwise.io/job/115699-it-systemadministrator-m-w-d",
    "https://www.workwise.io/job/113619-systemadministrator-fuer-microsoft-azure-cloud-architektur-m-w-d",
    "https://www.workwise.io/job/110963-senior-system-engineer-fuer-microsoft-azure-m-w-d",
    "https://www.workwise.io/job/109678-ki-ingenieur-m-w-d",
    "https://www.workwise.io/job/40204-systemadministrator-m-w-d",
    "https://www.workwise.io/job/109067-systems-architect-fuer-microsoft-m365-loesungen-m-w-d",
    "https://www.workwise.io/job/108714-senior-crm-softwareentwickler-m-w-d",
    "https://www.workwise.io/job/108712-projektmanager-fuer-digitalvertrieb-und-systemintegration-m-w-d",
    "https://www.workwise.io/job/109749-software-architekt-m-w-d",
    "https://www.workwise.io/job/114981-teamleitung-fuer-it-systemadministration-m-w-d",
    "https://www.workwise.io/job/110097-soc-architekt-fuer-cyber-security-m-w-d",
    "https://www.workwise.io/job/113808-senior-cloud-engineer-cloud-architect-m-w-d",
    "https://www.workwise.io/job/110551-project-funding-manager-m-w-d",
    "https://www.workwise.io/job/113229-analytics-engineer-fuer-data-warehouse-m-w-d",
    "https://www.workwise.io/job/25770-consultant-data-analytics-m-w-d",
    "https://www.workwise.io/job/25923-junior-consultant-data-analytics-m-w-d",
    "https://www.workwise.io/job/115246-senior-digital-marketing-consultant-fuer-strategische-sea-m-w-d",
    "https://www.workwise.io/job/115821-senior-produktmanager-fuer-cloud-loesungen-m-w-d",
    "https://www.workwise.io/job/55873-customer-data-insights-analyst-im-retail-m-w-d",
    "https://www.workwise.io/job/99749-weiterbildung-zum-data-analyst-fuer-quereinsteiger-mit-bildungsgutschein-100-online-m-w-d",
    "https://www.workwise.io/job/100509-data-analyst-im-private-equity-umfeld-m-w-d",
    "https://www.workwise.io/job/112005-werkstudent-als-financial-analyst-backoffice-assistent-m-w-d",
    "https://www.workwise.io/job/113501-process-analyst-w-m-d",
    "https://www.workwise.io/job/114995-experte-kartenzahlung-m-w-d",
    "https://www.workwise.io/job/90598-sap-application-consultant-fuer-energy-utilities-remote-m-w-d",
    "https://www.workwise.io/job/110367-dualer-master-junior-projektmanager-digitalisierung-energiemanagement-w-m-d",
    "https://www.workwise.io/job/60765-junior-consultant-fuer-sap-abap-entwicklung-m-w-d",
    "https://www.workwise.io/job/9580-junior-sap-softwareentwickler-m-w-d",
    "https://www.workwise.io/job/108384-anwendungsentwickler-fuer-ibm-system-m-w-d",
    "https://www.workwise.io/job/7819-werkstudent-im-bereich-systemadministration-softwareentwicklung-m-w-d",
    "https://www.workwise.io/job/114655-softwareentwickler-fuer-maschinensteuerung-m-w-d",
    "https://www.workwise.io/job/10963-werkstudent-software-testing-qualitaetsmanagement-m-w-d",
    "https://www.workwise.io/job/104792-datenbankadministrator-m-w-d",
    "https://www.workwise.io/job/115445-softwareentwickler-fuer-die-webentwicklung-m-w-d",
    "https://www.workwise.io/job/115746-oracle-apex-spezialist-fuer-anwendungsentwicklung-m-w-d",
    "https://www.workwise.io/job/107282-developer-and-system-administrator-m-w-d",
    "https://www.workwise.io/job/111874-netzwerk-und-systemadministrator-m-w-d",
    "https://www.workwise.io/job/109446-erp-systemadministrator-fuer-sage-b7c-m-w-d",
    "https://www.workwise.io/job/115471-qa-analyst-m-w-d",
    "https://www.workwise.io/job/33148-praktikant-data-science-data-engineering-im-bereich-risk-advisory-services",
    "https://www.workwise.io/job/54350-principal-it-consultant-fuer-strategische-java-und-cloud-systeme-m-w-d",
    "https://www.workwise.io/job/101057-software-developer-kotlin-java-sql-m-w-d",
    "https://www.workwise.io/job/110742-senior-python-backend-developer-w-m-d",
    "https://www.workwise.io/job/115218-it-systemadministrator-m-w-d-devops",
    "https://www.workwise.io/job/113919-senior-software-engineer-fuer-python-entwicklung-m-w-d",
    "https://www.workwise.io/job/110689-duales-studium-wirtschaftsinformatik-koeln-data-engineering-m-w-d",
    "https://www.workwise.io/job/108021-software-entwickler-fuer-clean-tech-automatisierung-m-w-d",
    "https://www.workwise.io/job/110740-senior-fullstack-entwickler-w-m-d",
    "https://www.workwise.io/job/112582-softwareentwickler-fuer-frontend-m-w-d",
    "https://www.workwise.io/job/115377-werkstudent-power-bi-m-w-d",
    "https://www.workwise.io/job/106991-soc-engineer-m-w-d",
    "https://www.workwise.io/job/61284-data-engineer-im-sap-umfeld-mit-beratungsfunktion-m-w-d",
    "https://www.workwise.io/job/112555-leiter-fuer-bare-metal-services-m-w-d",
    "https://www.workwise.io/job/25924-application-developer-oracle-cloud-m-w-d",
    "https://www.workwise.io/job/60112-cloud-systeme-consultant-fuer-agile-entwicklung-m-w-d",
    "https://www.workwise.io/job/109458-senior-consultant-intelligent-information-management-m-w-d",
    "https://www.workwise.io/job/115209-app-entwicklung-mit-ios-m-w-d",
    "https://www.workwise.io/job/6952-it-security-consultant-mit-schwerpunkt-it-infrastruktur-m-w-x",
    "https://www.workwise.io/job/37936-digital-infrastructure-expert-w-m-d",
    "https://www.workwise.io/job/104015-embedded-systems-entwickler-mit-c-c-kenntnissen-m-w-d",
    "https://www.workwise.io/job/113044-schichtfuehrer-fuer-sensormontage-und-personalkoordination-m-w-d",
    "https://www.workwise.io/job/104978-it-security-consultant-mit-schwerpunkt-isms-m-w-x",
    "https://www.workwise.io/job/109964-embedded-hard-and-software-developer-m-w-d",
    "https://www.workwise.io/job/114224-digital-platform-architect-m-w-d",
    "https://www.workwise.io/job/111298-golang-entwickler-fuer-virtual-private-server-m-w-d",
    "https://www.workwise.io/job/115400-praktikum-fuer-cloudbasierte-webapplikationen-m-w-d",
    "https://www.workwise.io/job/114664-werkstudent-fuer-medical-data-curation-m-w-d",
    "https://www.workwise.io/job/61274-it-consultant-mit-dem-schwerpunkt-datenintegration-m-w-d",
    "https://www.workwise.io/job/114059-full-stack-developer-m-w-d",
    "https://www.workwise.io/job/73786-it-consultant-cloud-telefonie-m-w-d",
    "https://www.workwise.io/job/45450-sap-junior-consultant-exa-solutions-m-w-d",
    "https://www.workwise.io/job/114088-prozessmanager-im-kreditgeschaeft-m-w-d",
    "https://www.workwise.io/job/107697-finance-associate-m-w-d",
    "https://www.workwise.io/job/113445-it-system-engineer-fuer-microsoft-technologien-m-w-d",
    "https://www.workwise.io/job/101739-ai-software-engineer-all-genders",
    "https://www.workwise.io/job/114069-senior-technical-artist-unity-and-xr-engineer-f-m-d",
    "https://www.workwise.io/job/109289-learning-solution-consultant-dach-m-w-d",
    "https://www.workwise.io/job/21535-senior-sap-bw-bi-consultant-m-w-d",
    "https://www.workwise.io/job/104791-it-systemadministrator-m-w-d",
    "https://www.workwise.io/job/105818-dualer-master-ai-ingenieur-fuer-genai-projekte-m-w-d",
    "https://www.workwise.io/job/68220-junior-it-consultant-in-nuernberg-bereich-smart-factory-industrie-40-m-w-d",
    "https://www.workwise.io/job/68216-software-developer-fullstack-in-nuernberg-net-c-industrie-40-m-w-d",
    "https://www.workwise.io/job/108018-weiterbildung-fachkraft-fuer-logistik-40-ihk-100-online-m-w-d",
    "https://www.workwise.io/job/103684-automatisierungstechniker-in-der-industrie-m-w-d",
    "https://www.workwise.io/job/112942-senior-mechanical-design-ingenieur-im-sondermaschinenbau-m-w-d",
    "https://www.workwise.io/job/102176-backend-entwickler-fuer-service-apis-m-w-d",
    "https://www.workwise.io/job/101206-senior-consultant-vertrieb-wohnungswirtschaft-m-w-d",
    "https://www.workwise.io/job/62894-mitarbeiter-im-vertrieb-sales-m-w-d",
    "https://www.workwise.io/job/113175-werksstudent-vertrieb-marketing-in-der-chemischen-industrie-m-w-d",
    "https://www.workwise.io/job/112888-werkstudent-fuer-den-vertrieb-im-business-development-m-w-d",
    "https://www.workwise.io/job/88759-pflichtpraktikant-im-vertrieb-sales-mit-fokus-auf-kundenakquise-m-w-d",
    "https://www.workwise.io/job/79452-aussendienstmitarbeiter-im-vertrieb-von-fahrzeugeinrichtungen-m-w-d",
    "https://www.workwise.io/job/113438-aussendienstmitarbeiter-im-vertrieb-m-w-d-sued-ost",
    "https://www.workwise.io/job/110185-teamlead-sales-im-b2b-vertrieb-m-w-d",
    "https://www.workwise.io/job/110114-vertriebsingenieur-in-der-blechverarbeitung-m-w-d",
    "https://www.workwise.io/job/114138-verkaufsberater-in-der-badausstellung-m-w-d",
    "https://www.workwise.io/job/110036-customer-service-assistent-im-key-account-m-w-d",
    "https://www.workwise.io/job/113254-kundenberater-im-internationalen-vertrieb-mit-fokus-frankreich-m-w-d",
    "https://www.workwise.io/job/94092-mitarbeiter-im-kundensupport-fuer-software-m-w-d",
    "https://www.workwise.io/job/109626-mitarbeiter-fuer-kundenbetreuung-beratung-m-w-d",
    "https://www.workwise.io/job/114090-werkstudent-m-w-d-im-technischen-kundendienst",
    "https://www.workwise.io/job/109542-assistenz-fuer-vertrieb-und-recruiting-m-w-d",
    "https://www.workwise.io/job/113682-assistenz-fuer-vertrieb-m-w-d",
    "https://www.workwise.io/job/108603-assistenz-im-vertragsmanagement-schwerpunkt-gesellschaftsrecht-m-w-d-in-voll-oder-teilzeit",
    "https://www.workwise.io/job/108753-vertriebsmitarbeiter-im-aussendienst-einzelhandel-m-w-d",
    "https://www.workwise.io/job/106390-technischer-vertriebsmitarbeiter-im-aussendienst-m-w-in-berlin",
    "https://www.workwise.io/job/98886-kundenberater-im-aussendienst-fuer-b2b-vertrieb-m-w-d-plz-83-84",
    "https://www.workwise.io/job/115688-sales-consultant-im-aussendienst-m-w-d",
    "https://www.workwise.io/job/106431-kaufmaennischer-mitarbeiter-aussendienst-kundenservice-m-w-d",
    "https://www.workwise.io/job/108533-vertriebsmitarbeiter-im-aussendienst-fuer-baumaschinen-m-w-d",
    "https://www.workwise.io/job/115492-it-sales-manager-m-w-d-region-hessen-it-systemhaus-kmu-mittelstand",
    "https://www.workwise.io/job/114984-teamleiter-customer-service-m-w-d",
    "https://www.workwise.io/job/114517-senior-customer-success-manager-consultant-m-w-d",
    "https://www.workwise.io/job/114545-social-media-und-e-commerce-koordinator-m-w-d",
    "https://www.workwise.io/job/112600-mitarbeiter-im-customer-service-beratung-m-w-d",
    "https://www.workwise.io/job/111886-werkstudent-customer-experience-support-m-w-d",
    "https://www.workwise.io/job/113870-sachbearbeiter-kundendienst-logistik-m-w-d",
    "https://www.workwise.io/job/113744-mitarbeiter-fuer-den-telefonischen-vertrieb-im-innendienst-m-w-d",
    "https://www.workwise.io/job/115006-spengler-fuer-bauprojekte-und-sanierung-m-w-d",
    "https://www.workwise.io/job/114976-business-development-manager-m-w-d",
    "https://www.workwise.io/job/115417-junior-sales-engineer-m-w-d-verkaufsgebiet-grossraum-muenchen",
    "https://www.workwise.io/job/114873-vertriebsmitarbeiter-fuer-kundenakquise-m-w-d",
    "https://www.workwise.io/job/115161-personaldisponent-m-w-d-mit-schwerpunkt-vertrieb-recruiting",
    "https://www.workwise.io/job/107123-praktikum-im-sales-mit-fokus-kundenakquise-m-w-d",
    "https://www.workwise.io/job/109418-kundenbetreuer-fuer-bestandskunden-m-w-d",
    "https://www.workwise.io/job/115531-kundenbetreuer-im-innendienst-m-w-d-in-teilzeit-quereinsteiger-willkommen",
    "https://www.workwise.io/job/93454-kundenbetreuer-fuer-laufende-mandantenbetreuung-m-w-d",
    "https://www.workwise.io/job/105325-account-executive-new-business-m-w-d",
    "https://www.workwise.io/job/108277-account-executive-fuer-die-neukundenakquise-m-w-d",
    "https://www.workwise.io/job/113931-account-executive-im-b2b-sales-m-w-d",
    "https://www.workwise.io/job/113939-key-account-manager-im-international-e-commerce-m-w-d",
    "https://www.workwise.io/job/112990-junior-account-manager-healthcare-pr-m-w-d",
    "https://www.workwise.io/job/105810-finanzbuchhalter-m-w-d",
    "https://www.workwise.io/job/114662-bilanzbuchhalter-accountant-m-w-d",
    "https://www.workwise.io/job/109777-teamlead-ffentliche-haushalte-rechnungswesen-m-w-d",
    "https://www.workwise.io/job/115653-finanzbuchhalter-im-bauwesen-m-w-d",
    "https://www.workwise.io/job/113724-buchhalter-steuerfachangestellter-im-buchhaltungsbuero-in-teilzeit-m-w-d",
    "https://www.workwise.io/job/112543-kaufmaennischer-mitarbeiter-in-der-abrechnung-m-w-d",
    "https://www.workwise.io/job/111943-personalsachbearbeiter-und-lohnabrechnung-m-w-d",
    "https://www.workwise.io/job/105205-bautechniker-projektleiter-m-w-d",
    "https://www.workwise.io/job/114800-junior-projektmanager-fuer-e-mobilitaet-m-w-d",
    "https://www.workwise.io/job/115460-assistenz-der-geschaeftsfuehrung-m-w-d",
    "https://www.workwise.io/job/101402-kaufmaennischer-sachbearbeiter-m-w-d-bueroorganisation",
    "https://www.workwise.io/job/105387-empfangsmitarbeiter-m-w-d-in-teilzeit",
    "https://www.workwise.io/job/114933-kaufmaennische-angestellte-fuer-die-buchhaltung-in-teilzeit-m-w-d",
    "https://www.workwise.io/job/113927-buerokauffrau-im-office-management-in-teilzeit-m-w-d",
    "https://www.workwise.io/job/113498-kaufmann-frau-im-gross-und-aussenhandel-mit-baustoffkenntnissen-m-w-d",
    "https://www.workwise.io/job/114947-office-manager-als-werkstudententaetigkeit-m-w-d",
    "https://www.workwise.io/job/115623-office-manager-m-w-d",
    "https://www.workwise.io/job/114866-empfangsmitarbeiter-fuer-bueroorganisation-m-w-d",
    "https://www.workwise.io/job/114971-kreditorenbuchhalter-m-w-d",
    "https://www.workwise.io/job/115678-sachbearbeiter-m-w-d-finanzbuchhaltung",
    "https://www.workwise.io/job/51645-pruefungsassistent-fuer-wirtschaftspruefung-und-steuern-m-w-d",
    "https://www.workwise.io/job/110356-kaufmaennischer-sachbearbeiter-im-anlagenbau-m-w-d",
    "https://www.workwise.io/job/70453-pruefungsassistent-fuer-wirtschaftspruefung-m-w-d",
    "https://www.workwise.io/job/87107-pruefungsassistent-fuer-jahresabschlusspruefungen-m-w-d",
    "https://www.workwise.io/job/94698-pruefungsassistent-wirtschaftspruefung-m-w-d",
    "https://www.workwise.io/job/114609-experte-fuer-regulatorik-und-wertpapier-compliance-befristet-m-w-d",
    "https://www.workwise.io/job/115057-schlosser-fuer-metallbau-und-konstruktion-m-w-d",
    "https://www.workwise.io/job/115391-finanzbuchhalter-fuer-hgb-abschluesse-m-w-d",
    "https://www.workwise.io/job/114158-finanzbuchhalter-fuer-debitoren-und-kreditoren-m-w-d",
    "https://www.workwise.io/job/108830-debitorenbuchhalter-im-rechnungswesen-m-w-d",
    "https://www.workwise.io/job/106553-finanzbuchhalter-fuer-debitoren-kreditorenbuchhaltung-m-w-d",
    "https://www.workwise.io/job/108099-debitorenbuchhalter-m-w-d",
    "https://www.workwise.io/job/114694-buchhalter-w-m-d-teilzeit-ab-20-stunden",
    "https://www.workwise.io/job/107016-finanzbuchhalter-fuer-zahlungsverkehr-m-w-d",
    "https://www.workwise.io/job/104691-leiter-finanzbuchhaltung-finanzbuchhalter-m-w-d",
    "https://www.workwise.io/job/114883-buchhalter-m-w-d",
    "https://www.workwise.io/job/108128-lohnbuchhalter-in-der-steuerkanzlei-m-w-d",
    "https://www.workwise.io/job/112241-finanzbuchhalter-m-w-d-in-teilzeit-15-20-std",
    "https://www.workwise.io/job/113857-sachbearbeitung-in-der-buchhaltung-w-m-d-ab-35-std",
    "https://www.workwise.io/job/110302-gruppenleiter-produktion-und-prozessoptimierung-m-w-d",
    "https://www.workwise.io/job/114940-bereichsleiter-produktion-in-grafenwald-m-w-d",
    "https://www.workwise.io/job/112653-produktionsmitarbeiter-m-w-d",
    "https://www.workwise.io/job/115354-abteilungsleiter-fuer-produktion-und-formteile-m-w-d",
    "https://www.workwise.io/job/112667-cnc-maschineneinrichter-mit-schichtleitung-in-der-produktion-m-w-d",
    "https://www.workwise.io/job/111908-produktionsmitarbeiter-in-der-lackherstellung-m-w-d",
    "https://www.workwise.io/job/115358-fachkraft-fuer-lagerlogistik-fuer-exportlager-m-w-d",
    "https://www.workwise.io/job/104795-staplerfahrer-fuer-hochregal-m-w-d",
    "https://www.workwise.io/job/114207-lagerist-fahrer-und-hausmeister-im-handwerk-m-w-d",
    "https://www.workwise.io/job/114718-projekt-staplerfahrer-fuer-systemtests-m-w-d",
    "https://www.workwise.io/job/112318-fachkraft-fuer-lagerlogistik-m-w-d",
    "https://www.workwise.io/job/113112-fachkraft-lagerlogistik-m-w-d-materialfluss-und-kaufmaennische-prozesse",
    "https://www.workwise.io/job/113335-einkaeufer-im-tief-und-strassenbau-m-w-d",
    "https://www.workwise.io/job/114884-supply-chain-manager-m-w-d",
    "https://www.workwise.io/job/111492-sachbearbeiter-einkauf-m-w-d",
    "https://www.workwise.io/job/115529-strategischer-einkaeufer-im-supply-chain-management-m-w-d",
    "https://www.workwise.io/job/114847-lagerleiter-fuer-materialbeschaffung-in-lingen-m-w-d",
    "https://www.workwise.io/job/78445-ausbildung-zum-kaufmann-im-gross-und-aussenhandelsmanagement-m-w-d",
    "https://www.workwise.io/job/113194-supply-chain-manager-fuer-einkauf-und-prozesse-m-w-d",
    "https://www.workwise.io/job/9260-technischer-einkaeufer-m-w-d",
    "https://www.workwise.io/job/114699-sachbearbeitung-m-w-d-supply-chain-management",
]

len(job_urls)

609

Web Scraping Results

- 609 job postings were scraped from workwise.io (manual list of URLs).
- For each posting, the HTML was converted into four text blocks:
  - `title`: Job title
  - `description`: Job description
  - `requirements`: Requirements/qualifications
  - `tasks`: Responsibilities/duties

## 2. Method A: Classic HTML scraping

Classic HTML scraping (requests + BeautifulSoup) attempts to extract individual content sections using stable selectors (e.g., description, requirements, tasks).

In [72]:
# Scraping a single job listing
def scrape_job(url): # Scrapes a single Workwise job entry, returns a dictionary with text fields
    headers = {"User-Agent": "Mozilla/5.0"}

    response = requests.get(url, headers=headers)
    if response.status_code != 200:
        return None

    soup = BeautifulSoup(response.text, "html.parser")

    # Title
    title_el = soup.find("h1")
    title = title_el.get_text(strip=True) if title_el else None

    # Description
    desc_el = soup.find("div", {"data-cy": "job-description"})
    description = desc_el.get_text(" ", strip=True) if desc_el else ""

    # Requirements
    req_el = soup.find("div", {"data-cy": "job-requirements"})
    requirements = req_el.get_text(" ", strip=True) if req_el else ""

    # Additional sections
    tasks_el = soup.find("div", {"data-cy": "job-tasks"})
    tasks = tasks_el.get_text(" ", strip=True) if tasks_el else ""

    return {
        "url": url,
        "title": title,
        "description": description,
        "requirements": requirements,
        "tasks": tasks,
    }

In [73]:
# Scrape all URLs
scraped_data = []

for url in job_urls[:10]: # Test: only the first 10 lines
    data = scrape_job(url)
    if data:
        scraped_data.append(data)

    time.sleep(2)

len(scraped_data)

10

In [74]:
# Create a DataFrame
df_raw_scrape = pd.DataFrame(scraped_data)
df_raw_scrape.head(10)

,url,title,description,requirements,tasks
0,https://www.workwise.io/job/53323-elektroniker...,Elektroniker als Servicetechniker (m/w/d),,,
1,https://www.workwise.io/job/77464-bilanzbuchha...,Bilanzbuchhalter mit Fokus Steuererklärung (m/...,,,
2,https://www.workwise.io/job/110163-bauingenieu...,Bauingenieur (m/w/d),,,
3,https://www.workwise.io/job/112719-pflegefachk...,Pflegefachkraft (m/w/d) für den Nachtdienst in...,,,
4,https://www.workwise.io/job/112918-steuerberat...,Steuerberater für mittelständische Unternehmen...,,,
5,https://www.workwise.io/job/113620-augenoptike...,Augenoptikermeister für Kundenberatung (m/w/d),,,
6,https://www.workwise.io/job/113675-kalkulator-...,Kalkulator im Gleisbau (m/w/d),,,
7,https://www.workwise.io/job/113947-kfz-mechatr...,Kfz-Mechatroniker für Nutzfahrzeuge (m/w/d),,,
8,https://www.workwise.io/job/114195-assistenz-d...,Assistenz der Geschäftsleitung in der Immobili...,,,
9,https://www.workwise.io/job/114248-anlagenmech...,Anlagenmechaniker für Heizungsanlagen (m/w/d),,,


Minimal Cleaning

- Free text
- Whitespace normalization
- Tokenization after cleaning
- Raw text = the most important foundation for skill mining

In [75]:
def clean_text(s):
    if pd.isna(s):
        return ""
    s = s.replace("\n", " ").replace("\r", " ")
    s = " ".join(s.split())
    return s

for col in ["title", "description", "requirements", "tasks"]:
    df_raw_scrape[col] = df_raw_scrape[col].astype(str).apply(clean_text)

df_raw_scrape.head(10)

,url,title,description,requirements,tasks
0,https://www.workwise.io/job/53323-elektroniker...,Elektroniker als Servicetechniker (m/w/d),,,
1,https://www.workwise.io/job/77464-bilanzbuchha...,Bilanzbuchhalter mit Fokus Steuererklärung (m/...,,,
2,https://www.workwise.io/job/110163-bauingenieu...,Bauingenieur (m/w/d),,,
3,https://www.workwise.io/job/112719-pflegefachk...,Pflegefachkraft (m/w/d) für den Nachtdienst in...,,,
4,https://www.workwise.io/job/112918-steuerberat...,Steuerberater für mittelständische Unternehmen...,,,
5,https://www.workwise.io/job/113620-augenoptike...,Augenoptikermeister für Kundenberatung (m/w/d),,,
6,https://www.workwise.io/job/113675-kalkulator-...,Kalkulator im Gleisbau (m/w/d),,,
7,https://www.workwise.io/job/113947-kfz-mechatr...,Kfz-Mechatroniker für Nutzfahrzeuge (m/w/d),,,
8,https://www.workwise.io/job/114195-assistenz-d...,Assistenz der Geschäftsleitung in der Immobili...,,,
9,https://www.workwise.io/job/114248-anlagenmech...,Anlagenmechaniker für Heizungsanlagen (m/w/d),,,


Result: Workwise loads the content of individual job postings (description, requirements, profile, skills) via JavaScript.
A simple requests call only sees the empty HTML shell, so the “description,” “requirements,” and “tasks” fields remain empty in the first scraping attempt. This makes it rather unsuitable as a dataset.

In [76]:
# Convert to the Unified Schema
def unify_scraped(df):
    df_unified = pd.DataFrame({
        "doc_id": [str(uuid.uuid4()) for _ in range(len(df))],
        "source_type": "job_ad_scrape",
        "source_name": "workwise_scrape",
        "job_title_raw": df["title"],
        "raw_text": (
            df["title"].astype(str) + " " +
            df["description"].astype(str) + " " +
            df["requirements"].astype(str) + " " +
            df["tasks"].astype(str)
        ),
        "language": "de", # Set to "de" retroactively, so the output is still "None"
        "meta_json": df.apply(
            lambda row: json.dumps({
                "url": row["url"],
                "requirements": row["requirements"],
                "tasks": row["tasks"]
            }),
            axis=1
        )
    })

    return df_unified

df_scrape_unified = unify_scraped(df_raw_scrape)
df_scrape_unified.head(10)

,doc_id,source_type,source_name,job_title_raw,raw_text,language,meta_json
0,c70c8789-c14a-447f-b4a3-283546c402cc,job_ad_scrape,workwise_scrape,Elektroniker als Servicetechniker (m/w/d),Elektroniker als Servicetechniker (m/w/d),de,"{""url"": ""https://www.workwise.io/job/53323-ele..."
1,3e193ea8-0ab4-4488-95a4-ccbec5d43b63,job_ad_scrape,workwise_scrape,Bilanzbuchhalter mit Fokus Steuererklärung (m/...,Bilanzbuchhalter mit Fokus Steuererklärung (m/...,de,"{""url"": ""https://www.workwise.io/job/77464-bil..."
2,6e954a37-385f-4b3f-a2ad-33f3e9b5375e,job_ad_scrape,workwise_scrape,Bauingenieur (m/w/d),Bauingenieur (m/w/d),de,"{""url"": ""https://www.workwise.io/job/110163-ba..."
3,128e7da4-718b-4c9f-a025-43d0585af1cb,job_ad_scrape,workwise_scrape,Pflegefachkraft (m/w/d) für den Nachtdienst in...,Pflegefachkraft (m/w/d) für den Nachtdienst in...,de,"{""url"": ""https://www.workwise.io/job/112719-pf..."
4,efa36159-64ad-4d1e-87e7-819684b82ad1,job_ad_scrape,workwise_scrape,Steuerberater für mittelständische Unternehmen...,Steuerberater für mittelständische Unternehmen...,de,"{""url"": ""https://www.workwise.io/job/112918-st..."
5,2575e80e-f110-43b7-83e3-254713e2d922,job_ad_scrape,workwise_scrape,Augenoptikermeister für Kundenberatung (m/w/d),Augenoptikermeister für Kundenberatung (m/w/d),de,"{""url"": ""https://www.workwise.io/job/113620-au..."
6,30268147-0ec9-4f4f-858b-31a22117ed56,job_ad_scrape,workwise_scrape,Kalkulator im Gleisbau (m/w/d),Kalkulator im Gleisbau (m/w/d),de,"{""url"": ""https://www.workwise.io/job/113675-ka..."
7,96f23b54-6eb8-449c-b910-dc539d2d3958,job_ad_scrape,workwise_scrape,Kfz-Mechatroniker für Nutzfahrzeuge (m/w/d),Kfz-Mechatroniker für Nutzfahrzeuge (m/w/d),de,"{""url"": ""https://www.workwise.io/job/113947-kf..."
8,7e0472ec-709f-4e62-a2d3-1b946523e6c3,job_ad_scrape,workwise_scrape,Assistenz der Geschäftsleitung in der Immobili...,Assistenz der Geschäftsleitung in der Immobili...,de,"{""url"": ""https://www.workwise.io/job/114195-as..."
9,46c9a4ff-add3-47ea-a392-d4dea49be0e9,job_ad_scrape,workwise_scrape,Anlagenmechaniker für Heizungsanlagen (m/w/d),Anlagenmechaniker für Heizungsanlagen (m/w/d),de,"{""url"": ""https://www.workwise.io/job/114248-an..."


Unified Document Schema

- The scraped job postings are transferred to the project-wide Unified Document Schema:
  - `job_title_raw` = Job title
  - `raw_text` = Title + Description + Requirements + Tasks in a free-text field
  - `source_type = “job_ad_scrape”`, `source_name = “workwise_scrape”`
  - `meta_json` additionally contains: `url`, `requirements`, `tasks`
- This makes the Workwise ads fully compatible with the rest of the database and allows them to be used later with the same methods.

In [77]:
# Save as Parquet
output_path = DATA_INTERIM_EXTERNAL / "scraped_workwise_unified.parquet"
df_scrape_unified.to_parquet(output_path)

print("Gespeichert unter:", output_path)

Gespeichert unter: C:\Users\sigle\OneDrive - Hochschule Reutlingen\Dokumente\Profilerweiterung\data\interim_external\scraped_workwise_unified.parquet


In [78]:
# Quality Check
df_scrape_unified.info()
df_scrape_unified[['job_title_raw', 'raw_text']].head(10)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 7 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   doc_id         10 non-null     object
 1   source_type    10 non-null     object
 2   source_name    10 non-null     object
 3   job_title_raw  10 non-null     object
 4   raw_text       10 non-null     object
 5   language       10 non-null     object
 6   meta_json      10 non-null     object
dtypes: object(7)
memory usage: 692.0+ bytes


,job_title_raw,raw_text
0,Elektroniker als Servicetechniker (m/w/d),Elektroniker als Servicetechniker (m/w/d)
1,Bilanzbuchhalter mit Fokus Steuererklärung (m/...,Bilanzbuchhalter mit Fokus Steuererklärung (m/...
2,Bauingenieur (m/w/d),Bauingenieur (m/w/d)
3,Pflegefachkraft (m/w/d) für den Nachtdienst in...,Pflegefachkraft (m/w/d) für den Nachtdienst in...
4,Steuerberater für mittelständische Unternehmen...,Steuerberater für mittelständische Unternehmen...
5,Augenoptikermeister für Kundenberatung (m/w/d),Augenoptikermeister für Kundenberatung (m/w/d)
6,Kalkulator im Gleisbau (m/w/d),Kalkulator im Gleisbau (m/w/d)
7,Kfz-Mechatroniker für Nutzfahrzeuge (m/w/d),Kfz-Mechatroniker für Nutzfahrzeuge (m/w/d)
8,Assistenz der Geschäftsleitung in der Immobili...,Assistenz der Geschäftsleitung in der Immobili...
9,Anlagenmechaniker für Heizungsanlagen (m/w/d),Anlagenmechaniker für Heizungsanlagen (m/w/d)


## 3. Method B: Dynamic Selenium Scraping

Method A extracts only statically visible HTML text. However, Workwise loads most of its content dynamically (React/Next.js) --> therefore, browser-based scraping is necessary (Dai et al. 2015).

In [79]:
# Selenium Setup
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.chrome.service import Service
from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

chrome_options = Options()
chrome_options.add_argument("--headless")
chrome_options.add_argument("--no-sandbox")
chrome_options.add_argument("--disable-dev-shm-usage")

#driver = webdriver.Chrome(options=chrome_options)
driver = webdriver.Chrome(
    service=Service(ChromeDriverManager().install()),
    options=chrome_options
)
driver.set_page_load_timeout(30)

Function for dynamic scraping (Selenium) to retrieve all visible text at once:

In [80]:
from selenium.common.exceptions import TimeoutException
from bs4 import BeautifulSoup

def scrape_dynamic_job(url): # Dynamic scraping of a Workwise job posting, retrieving the title and the body text (description, responsibilities, requirements, etc.)

    try:
        # Load page
        driver.get(url)
    except TimeoutException:
        return None
    except Exception:
        return None
    
    # Waiting for the headline
    try:
        WebDriverWait(driver, 10).until(
            EC.presence_of_element_located((By.TAG_NAME, "h1"))
        )
    except TimeoutException:
        return None

    # Title
    try:
        title = driver.find_element(By.TAG_NAME, "h1").text.strip()
    except Exception:
        title = ""

    # Fetch the entire page as HTML and parse it using BeautifulSoup
    html = driver.page_source
    soup = BeautifulSoup(html, "html.parser")

    # Get content sections
    parts = []
    for cy in ["job-description", "job-tasks", "job-requirements"]:
        el = soup.find("div", {"data-cy": cy})
        if el:
            parts.append(el.get_text(separator="\n", strip=True))

    # Fallback: Main container of the display or body
    if not parts:
        # take the entire job detail container
        container = soup.find("div", {"data-cy": "job-detail-page"})
        if container:
            text = container.get_text(separator="\n", strip=True)
        else:
            body = soup.find("body")
            text = body.get_text(separator="\n", strip=True) if body else ""

        parts = [text]

    full_text = "\n\n".join(parts)

    # Trim navigation, remove unnecessary information
    if title:
        idx = full_text.find(title)
        if idx != -1:
            full_text = full_text[idx:]

    # Remove everything after "FAQs/Similar Jobs/Footer", otherwise there will be too much information
    stop_markers = ["Häufige Fragen","Ähnliche Jobs für dich","Unser Jobangebot","Workwise Untermenü anzeigen","Workwise Untermenü"]
    for marker in stop_markers:
        pos = full_text.find(marker)
        if pos != -1:
            full_text = full_text[:pos]
            break

    full_text = full_text.strip()

    return {
        "url": url,
        "title": title,
        "raw_text": full_text, # Here, just use raw text; free-form text is sufficient instead of artificial divisions (into description, requirements, etc.)
    }

Loop to scrape all URLs (as in Method A): this version is more robust to ensure a complete run; invalid ads are logged and end up in the "errors" folder instead of causing the process to abort

In [81]:
scraped_dynamic = []
errors = []
empty = []

for i, url in enumerate(job_urls):   # job_urls = 600 URLs
# for i, url in enumerate(job_urls[:100]):  # test with 100 URLs
    print(f"{i+1}/{len(job_urls)}: {url}")
    
    try:
        data = scrape_dynamic_job(url)

        if data and data.get("raw_text", "").strip():
            scraped_dynamic.append(data)
        else:
            empty.append(url)

    except Exception as e:
        errors.append((url, repr(e)))

# Summary
print("SCRAPING ERGEBNISSE:")
print("Gesamt-URLs:", len(job_urls))
print("Erfolgreich:", len(scraped_dynamic))
print("Leer:", len(empty))
print("Fehler:", len(errors))

1/609: https://www.workwise.io/job/53323-elektroniker-als-servicetechniker-m-w-d
2/609: https://www.workwise.io/job/77464-bilanzbuchhalter-mit-fokus-steuererklaerung-m-w-d
3/609: https://www.workwise.io/job/110163-bauingenieur-m-w-d
4/609: https://www.workwise.io/job/112719-pflegefachkraft-m-w-d-fuer-den-nachtdienst-in-betzdorf
5/609: https://www.workwise.io/job/112918-steuerberater-fuer-mittelstaendische-unternehmen-m-w-d
6/609: https://www.workwise.io/job/113620-augenoptikermeister-fuer-kundenberatung-m-w-d
7/609: https://www.workwise.io/job/113675-kalkulator-im-gleisbau-m-w-d
8/609: https://www.workwise.io/job/113947-kfz-mechatroniker-fuer-nutzfahrzeuge-m-w-d
9/609: https://www.workwise.io/job/114195-assistenz-der-geschaeftsleitung-in-der-immobilienbranche-m-w-d
10/609: https://www.workwise.io/job/114248-anlagenmechaniker-fuer-heizungsanlagen-m-w-d
11/609: https://www.workwise.io/job/114644-polier-fuer-verkehrsstationen-m-w-d
12/609: https://www.workwise.io/job/114874-steuerfachange

In [82]:
# DataFrame
df_dynamic_raw = pd.DataFrame(scraped_dynamic)
df_dynamic_raw.head()

,url,title,raw_text
0,https://www.workwise.io/job/53323-elektroniker...,Elektroniker als Servicetechniker (m/w/d),Elektroniker als Servicetechniker (m/w/d)\nSch...
1,https://www.workwise.io/job/77464-bilanzbuchha...,Bilanzbuchhalter mit Fokus Steuererklärung (m/...,Bilanzbuchhalter mit Fokus Steuererklärung (m/...
2,https://www.workwise.io/job/110163-bauingenieu...,Bauingenieur (m/w/d),Bauingenieur (m/w/d)\nSchneller-Timer Symbol\n...
3,https://www.workwise.io/job/112719-pflegefachk...,Pflegefachkraft (m/w/d) für den Nachtdienst in...,Pflegefachkraft (m/w/d) für den Nachtdienst in...
4,https://www.workwise.io/job/112918-steuerberat...,Steuerberater für mittelständische Unternehmen...,Steuerberater für mittelständische Unternehmen...


Minimal Cleaning

In [83]:
import re
import unicodedata

def clean_text2(s):
    if pd.isna(s):
        return ""
    # Normalize Unicode
    s = unicodedata.normalize("NFKC", str(s))
    
    # Standardize line breaks
    s = s.replace("\r", " ").replace("\n", " ")
    
    # Remove UI elements (icons, navigation)
    ui_tokens = [" Symbol"," Icon"," Untermenü anzeigen"," Karte laden"," Standort laden",]
    for token in ui_tokens:
        s = s.replace(token, " ")
    
    # Multiple spaces as a single space
    s = re.sub(r"\s+", " ", s)
    
    return s.strip()

df_dynamic_raw["raw_text"] = df_dynamic_raw["raw_text"].apply(clean_text2)

Create a unified schema:

In [84]:
def unify_dynamic(df):
    return pd.DataFrame({
        "doc_id": [str(uuid.uuid4()) for _ in range(len(df))],
        "source_type": "job_ad_scrape",
        "source_name": "workwise_selenium",
        "job_title_raw": df["title"],
        "raw_text": df["raw_text"],
        "language": "de", # Set to "de" later, which is why the output is still "None" here
        "meta_json": df["url"].apply(lambda x: json.dumps({"url": x}))
    })

df_dynamic_unified = unify_dynamic(df_dynamic_raw)
df_dynamic_unified.head()

,doc_id,source_type,source_name,job_title_raw,raw_text,language,meta_json
0,1b5b9def-51ff-4cfd-9d53-b4b4ee3e3b0e,job_ad_scrape,workwise_selenium,Elektroniker als Servicetechniker (m/w/d),Elektroniker als Servicetechniker (m/w/d) Schn...,de,"{""url"": ""https://www.workwise.io/job/53323-ele..."
1,632e7b1e-e080-4f0e-85be-f1d9495a531b,job_ad_scrape,workwise_selenium,Bilanzbuchhalter mit Fokus Steuererklärung (m/...,Bilanzbuchhalter mit Fokus Steuererklärung (m/...,de,"{""url"": ""https://www.workwise.io/job/77464-bil..."
2,7c16b0f8-5e00-4cae-86d1-17b8d3d5ae6e,job_ad_scrape,workwise_selenium,Bauingenieur (m/w/d),Bauingenieur (m/w/d) Schneller-Timer Festanste...,de,"{""url"": ""https://www.workwise.io/job/110163-ba..."
3,0c003eb9-7407-4cf2-b950-10776f83ab14,job_ad_scrape,workwise_selenium,Pflegefachkraft (m/w/d) für den Nachtdienst in...,Pflegefachkraft (m/w/d) für den Nachtdienst in...,de,"{""url"": ""https://www.workwise.io/job/112719-pf..."
4,7ff85830-7d0d-429c-aece-b08313bc8a6b,job_ad_scrape,workwise_selenium,Steuerberater für mittelständische Unternehmen...,Steuerberater für mittelständische Unternehmen...,de,"{""url"": ""https://www.workwise.io/job/112918-st..."


In [85]:
# Save
output_path = DATA_INTERIM_EXTERNAL / "scraped_workwise_dynamic_unified.parquet"
df_dynamic_unified.to_parquet(output_path)

print("Gespeichert unter:", output_path)

Gespeichert unter: C:\Users\sigle\OneDrive - Hochschule Reutlingen\Dokumente\Profilerweiterung\data\interim_external\scraped_workwise_dynamic_unified.parquet


## 4. Comparison of Method A vs. Method B
- Method A (HTML) extracts only titles, hardly any descriptions; unsuitable. Workwise uses React/Next.js; content is rendered dynamically via JavaScript
- Method B (Selenium) provides complete free-form text:
    - Job responsibilities
    - Your profile
    - Knowledge & Skills, Languages
    - Requirements
    - Benefits

Method B is used because it yields suitable, promising results

## 5. Quality Checks

In [86]:
# Basic overview of the scraped dataset
df_dynamic_unified.info()
df_dynamic_unified[["job_title_raw", "raw_text"]].head(5)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 386 entries, 0 to 385
Data columns (total 7 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   doc_id         386 non-null    object
 1   source_type    386 non-null    object
 2   source_name    386 non-null    object
 3   job_title_raw  386 non-null    object
 4   raw_text       386 non-null    object
 5   language       386 non-null    object
 6   meta_json      386 non-null    object
dtypes: object(7)
memory usage: 21.2+ KB


,job_title_raw,raw_text
0,Elektroniker als Servicetechniker (m/w/d),Elektroniker als Servicetechniker (m/w/d) Schn...
1,Bilanzbuchhalter mit Fokus Steuererklärung (m/...,Bilanzbuchhalter mit Fokus Steuererklärung (m/...
2,Bauingenieur (m/w/d),Bauingenieur (m/w/d) Schneller-Timer Festanste...
3,Pflegefachkraft (m/w/d) für den Nachtdienst in...,Pflegefachkraft (m/w/d) für den Nachtdienst in...
4,Steuerberater für mittelständische Unternehmen...,Steuerberater für mittelständische Unternehmen...


Structure and sample rows:
- info() shows that all job postings contain a doc_id, a job title (job_title_raw), free text (raw_text), and metadata (meta_json with URL)
- The first few rows show that the free text contains the key content of the Workwise job postings (tasks, requirements, profile, benefits)
- The structure corresponds to the Unified Document Schema and is compatible with the other external datasets

In [87]:
df_dynamic_unified.loc[0, "raw_text"]

'Elektroniker als Servicetechniker (m/w/d) Schneller-Timer Festanstellung Standort Andernach Plus 38 Schwank GmbH Stern Stern Stern Stern Stern 3,6 (53 Bewertungen auf ) Bild ansehen Kalender Ab sofort gesucht (unbefristet) Uhr 40 h pro Woche Dollar 44.000–58.000 € pro Jahr (verhandelbar) Homeoffice Kein Homeoffice Eigentlich suchen wir Held:innen. Held:innen, die von unseren Kund:innen mit einem „Schön, dass Sie da sind.“ begrüßt werden. Denn als Teil unserer Service-Familie sorgst du dafür, dass unsere Kund:innen zufrieden sind. Verantwortlichkeiten Was erwartet dich? Du betreibst Problemanalysen, Instandhaltung und Störungsbehebung und stellst damit den reibungslosen Ablauf sicher Du stellst die Einsatzbereitschaft von Heiz- und Klimasystemen bei unseren Kunden sicher und gewährleistet damit deren Produktivität und Effizienz Du bekommst bei deinem Start eine gründliche Einarbeitung durch die erfahrenen Kolleg:innen Du stellst dich wechselnden Herausforderungen und löst individuelle 

Text Length/Word Count:

In [88]:
# Character and word counts of the texts
df_dynamic_unified["text_len"] = df_dynamic_unified["raw_text"].str.len()
df_dynamic_unified["word_count"] = df_dynamic_unified["raw_text"].str.split().str.len()

df_dynamic_unified[["text_len", "word_count"]].describe()

,text_len,word_count
count,386.000000,386.000000
mean,3518.360104,424.427461
std,813.686201,105.650151
min,419.000000,60.000000
25%,3005.250000,355.000000
50%,3422.000000,406.500000
75%,3990.750000,482.750000
max,7094.000000,935.000000


In [89]:
# Shortest
df_dynamic_unified[["job_title_raw", "word_count"]].sort_values("word_count").head(10)

,job_title_raw,word_count
144,Bad gateway Error code 502,60
145,Bad gateway Error code 502,60
146,Bad gateway Error code 502,60
238,Kfz-Mechaniker / Kfz-Mechatroniker m/w/d,205
136,Maler- und Lackierer Meister im Handwerk (m/w/d),240
240,Zuschneider im Bereich Flachglas (m/w/d),253
99,Zweiradmechaniker/KFZ-Mechatroniker für Servic...,258
312,Monteur für Sicherheitssysteme (m/w/d),267
101,Steuerfachangestellte für Finanzbuchhaltung (m...,268
127,Kfz-Mechatroniker in der Fahrzeugwartung (m/w/d),272


Length of job descriptions
- The distribution of text lengths (text_len, word_count) shows that most job postings contain several hundred words
- Sufficient free-form text to extract skills, requirements, and duties later
- Short texts are visible and could be filtered out if necessary

Complete examples of manual checks:

In [90]:
# Display a sample ad in its entirety
example = df_dynamic_unified.sample(1, random_state=42).iloc[0]

meta = json.loads(example["meta_json"])

print("URL:", meta["url"])
print("\nJOB TITLE:")
print(example["job_title_raw"])

print("\nRAW TEXT:")
print(example["raw_text"])

URL: https://www.workwise.io/job/110240-techniker-sanitaerhandwerk-w-m-d

JOB TITLE:
Techniker Sanitärhandwerk (w/m/d)

RAW TEXT:
Techniker Sanitärhandwerk (w/m/d) Schneller-Timer Festanstellung Standort München Das Unternehmen wird nach deiner Bewerbung sichtbar. Tooltip anzeigen Bild ansehen Kalender Ab sofort gesucht (unbefristet) Uhr 25–40 h pro Woche Dollar Kein Gehalt angegeben Homeoffice 1 - 2 Tage pro Woche Homeoffice Planung und Projektierung der technischen Gebäudeausrüstung mit Schwerpunkt "Trinkwasser" für Sanierungen in Bestandsimmobilien. Verantwortlichkeiten Was erwartet dich? Du arbeitest aktiv in allen Leistungsphasen mit Du übernimmst eine interessante und abwechslungsreiche Tätigkeit Du erlebst eine offene, durch echte Teamarbeit geprägte Arbeitsatmosphäre mit vielen Gestaltungsmöglichkeiten und viel Spaß bei der Arbeit Du arbeitest eigenverantwortlich und ergebnisorientiert Ohne Personalverantwortung Berufsfelder Anlagenmechaniker für Sanitär-, Heizungs- und Klimate

In [91]:
# Display a sample ad in its entirety
example = df_dynamic_unified.sample(2, random_state=42).iloc[1]

meta = json.loads(example["meta_json"])

print("URL:", meta["url"])
print("\nJOB TITLE:")
print(example["job_title_raw"])

print("\nRAW TEXT:")
print(example["raw_text"])

URL: https://www.workwise.io/job/109413-industriemechaniker-m-w-d

JOB TITLE:
Industriemechaniker (m/w/d)

RAW TEXT:
Industriemechaniker (m/w/d) Schneller-Timer Festanstellung Standort Alzey THIMM Alle 14 Bilder ansehen Kalender Ab sofort gesucht (unbefristet) Uhr 35–40 h pro Woche Dollar Kein Gehalt angegeben Homeoffice Kein Homeoffice Ob etwas in der Maschine klemmt oder eine Dichtung versagt – du findest die Ursache und die Lösung! Oft bringt dich dein logisches Denken weiter und schon hast du an der richtigen Schraube gedreht und das Problem ist gelöst. Du bist Troubleshooter:inund Maschinenpfleger:in, liebst vielseitige und abwechslungsreiche Herausforderungen. Ob Landmaschinen, Traktoren oder - wie bei uns - richtig große Industriemaschinen: du verstehst, wie sie laufen und was sie brauchen! Verantwortlichkeiten Was erwartet dich? Du übernimmst selbstständig die Durchführung von Wartungen und Instandhaltungsmaßnahmen innerhalb der Wartungsschichten sowie die Schichtbegleitung und

Manual plausibility check, random listing selected and displayed in full, comparison with the original page (open the Workwise URL in the browser):
- Core content elements (description, duties: “What can you expect?”, requirements: “What should you bring to the table?”, Benefits: “What do we offer you?”, Knowledge & Skills, Company Description (not critical for text quality, is ignored, may provide industry information)) are included in the free text
- Website formatting (icons, bullet points, layout) is lost as expected, not relevant
- Some additional website elements, such as cookie notices or UI components, are scraped along with the main content. Some elements have already been removed (headline, footer, etc.)
- Overall, the Workwise scrape can be classified as being of sufficient quality to be integrated as an additional job ad source for profile expansion.

The browser closes properly:

In [92]:
driver.quit()

# Conclusion: Notebook 06

- This notebook creates a scraped dataset containing 609 current job listings from workwise.io (https://www.workwise.io/jobs)
- Expansion of the database of purely German external datasets
- Free-text fields were extracted, lightly cleaned, and the data was converted to the Unified Document Schema.
- Overall, the Workwise scrape can be classified as being of sufficient quality to be integrated as an additional job ad source for the profile expansion. Especially since it represents another purely German-language source.